# Fine-tuning Gemma 2-2B on the CBU Faculty Manual
### Google Colab · T4 GPU · QLoRA · Unsloth · Ollama

This notebook fine-tunes a Gemma 2-2B language model on the California Baptist University
Employee Manual (Faculty Section) using QLoRA on a free-tier Google Colab T4 GPU.

**Pipeline overview**
```
PDF → Chunks → QA Pairs → Format → QLoRA Train → Evaluate → Export GGUF
  1       2        3         4           5            6           7
```

| Stage | Description | Output |
|-------|-------------|--------|
| 1 — Chunk | Parse PDF into policy sections | `data/chunks.jsonl` — ~90 chunks |
| 2 — Generate | QA pairs via Gemma 2 inference | `data/seeds.jsonl` — 300 pairs |
| 3 — Format | Apply Gemma 2 chat template | `data/train.jsonl` + `data/valid.jsonl` |
| 4 — Train | QLoRA fine-tuning via Unsloth | `outputs/cbu_gemma2_adapter/` |
| 5 — Evaluate | ROUGE-L: base vs fine-tuned | `eval/results.jsonl` |
| 6 — Plot | Per-type bar chart | `eval/rouge_chart.png` |
| 7 — Export | Q4_K_M GGUF for Ollama | `burns/model.Q4_K_M.gguf` |

**Prerequisites**
- Enable GPU: Runtime → Change runtime type → T4 GPU
- Accept Gemma 2 license at [huggingface.co/google/gemma-2-2b-it](https://huggingface.co/google/gemma-2-2b-it)
- A HuggingFace token with read access (free): [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
- `cbu_faculty.pdf` — upload when prompted in Step 3

All outputs are saved to Google Drive at `MyDrive/cbu-faculty-llm/` so they survive session resets.

---
## Step 0 — Install Packages

Run this cell once, then **do not restart the runtime** — Unsloth handles its own dependencies.

In [ ]:
%%capture
# Unsloth Colab-specific install (pins compatible trl version)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl==0.8.6" peft accelerate bitsandbytes
!pip install pdfplumber rouge-score matplotlib datasets
!pip install --upgrade "huggingface_hub[cli]"

---
## Step 1 — Verify GPU Environment

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not found — enable via Runtime → Change runtime type → T4 GPU"

gpu    = torch.cuda.get_device_name(0)
vram   = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16   = torch.cuda.is_bf16_supported()

print(f"GPU   : {gpu}")
print(f"VRAM  : {vram:.1f} GB")
print(f"BF16  : {'supported' if bf16 else 'not supported (will use FP16)'}")
print(f"Torch : {torch.__version__}")
print("\nEnvironment OK.")

---
## Step 2 — Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE   = '/content/drive/MyDrive/cbu-faculty-llm'
PDF_PATH     = '/content/cbu_faculty.pdf'
CHUNKS_PATH  = f'{DRIVE_BASE}/data/chunks.jsonl'
SEEDS_PATH   = f'{DRIVE_BASE}/data/seeds.jsonl'
TRAIN_PATH   = f'{DRIVE_BASE}/data/train.jsonl'
VALID_PATH   = f'{DRIVE_BASE}/data/valid.jsonl'
ADAPTER_PATH = f'{DRIVE_BASE}/outputs/cbu_gemma2_adapter'
EVAL_PATH    = f'{DRIVE_BASE}/eval/results.jsonl'
CHART_PATH   = f'{DRIVE_BASE}/eval/rouge_chart.png'
GGUF_DIR     = f'{DRIVE_BASE}/burns'
MODEL_NAME   = 'unsloth/gemma-2-2b-it'

for d in [f'{DRIVE_BASE}/data', f'{DRIVE_BASE}/outputs',
          f'{DRIVE_BASE}/eval',  f'{DRIVE_BASE}/burns']:
    os.makedirs(d, exist_ok=True)

free_gb = os.statvfs('/content/drive').f_bavail * os.statvfs('/content/drive').f_bsize / 1e9
print(f"Drive mounted. Free space: {free_gb:.1f} GB")
print(f"Working directory: {DRIVE_BASE}")

---
## Step 3 — Upload PDF & HuggingFace Login

Upload `cbu_faculty.pdf` when prompted. If it was uploaded in a previous session it will be loaded from Drive automatically.

In [ ]:
import shutil
from google.colab import files

DRIVE_PDF = f'{DRIVE_BASE}/cbu_faculty.pdf'

if os.path.exists(DRIVE_PDF):
    shutil.copy(DRIVE_PDF, PDF_PATH)
    size_mb = os.path.getsize(PDF_PATH) / 1e6
    print(f"PDF loaded from Drive ({size_mb:.1f} MB)")
else:
    print("Upload cbu_faculty.pdf:")
    uploaded = files.upload()
    for fname in uploaded:
        shutil.copy(fname, PDF_PATH)
        shutil.copy(fname, DRIVE_PDF)
    print(f"PDF saved to Drive.")

# HuggingFace login (required for Gemma 2 gated model)
from huggingface_hub import login
login()  # paste your HF token when prompted

---
## Step 4 — Extract PDF → Policy Chunks

The PDF is parsed into policy-level sections using the CBU handbook's own structure.
Policy boundaries are detected by scanning for standalone policy numbers (e.g. `3.102`, `3.300`)
on pages that contain `Effective Date:` headers — robust to CBU's varied PDF formatting.

Each chunk maps to one policy number and includes: title, category, effective date, and word count.

Expected output: **~90 chunks across 11 categories**

In [ ]:
import re
import json
import pdfplumber
from collections import Counter
from typing import Optional

# ── Policy map (title, section, category per policy number) ──
POLICY_MAP = {
    "3.000": {"title": "Introduction and Statement of Anti-Discrimination",        "section": "Foundation",           "category": "foundation"},
    "3.001": {"title": "Values of California Baptist University",                  "section": "Foundation",           "category": "foundation"},
    "3.002": {"title": "Mission and Philosophy",                                   "section": "Foundation",           "category": "foundation"},
    "3.003": {"title": "Statement of Faith",                                       "section": "Foundation",           "category": "foundation"},
    "3.004": {"title": "Academic Freedom and Responsibility",                      "section": "Foundation",           "category": "academic_freedom"},
    "3.005": {"title": "Intellectual Property Rights",                             "section": "Foundation",           "category": "academic_freedom"},
    "3.100": {"title": "Employment Overview",                                      "section": "Employment",           "category": "employment"},
    "3.101": {"title": "Recruitment",                                              "section": "Employment",           "category": "employment"},
    "3.102": {"title": "Faculty Appointments",                                     "section": "Employment",           "category": "employment"},
    "3.103": {"title": "Faculty Responsibilities",                                 "section": "Employment",           "category": "employment"},
    "3.104": {"title": "Faculty Roles",                                            "section": "Employment",           "category": "employment"},
    "3.105": {"title": "Faculty Awards",                                           "section": "Employment",           "category": "employment"},
    "3.106": {"title": "Load",                                                     "section": "Employment",           "category": "employment"},
    "3.107": {"title": "Termination",                                              "section": "Employment",           "category": "employment"},
    "3.108": {"title": "Emeritus Status",                                          "section": "Employment",           "category": "employment"},
    "3.109": {"title": "Adjunct Faculty: Definitions, Roles and Duties",           "section": "Employment",           "category": "employment"},
    "3.110": {"title": "Student Honor Code Violations",                            "section": "Employment",           "category": "conduct"},
    "3.200": {"title": "Promotion and Tenure Overview",                            "section": "Tenure and Promotion", "category": "tenure_promotion"},
    "3.201": {"title": "Promotion",                                                "section": "Tenure and Promotion", "category": "tenure_promotion"},
    "3.202": {"title": "Authority to Grant/Terminate Tenure",                      "section": "Tenure and Promotion", "category": "tenure_promotion"},
    "3.203": {"title": "Eligibility for Earning Tenure",                           "section": "Tenure and Promotion", "category": "tenure_promotion"},
    "3.204": {"title": "Qualifications for Earning Promotion and Tenure",          "section": "Tenure and Promotion", "category": "tenure_promotion"},
    "3.205": {"title": "Sequence of Procedures of Tenure",                         "section": "Tenure and Promotion", "category": "tenure_promotion"},
    "3.206": {"title": "Post Tenure Review",                                       "section": "Tenure and Promotion", "category": "tenure_promotion"},
    "3.207": {"title": "Annual Evaluation of Faculty and Merit Pay",               "section": "Tenure and Promotion", "category": "tenure_promotion"},
    "3.300": {"title": "Reduced Load",                                             "section": "Leave and Benefits",   "category": "leave_benefits"},
    "3.301": {"title": "Sabbatical",                                               "section": "Leave and Benefits",   "category": "leave_benefits"},
    "3.302": {"title": "Leaves of Absence",                                        "section": "Leave and Benefits",   "category": "leave_benefits"},
    "3.303": {"title": "Transitional Retirement",                                  "section": "Leave and Benefits",   "category": "leave_benefits"},
    "3.304": {"title": "Educational Assistance",                                   "section": "Leave and Benefits",   "category": "leave_benefits"},
    "3.305": {"title": "Sick Leave",                                               "section": "Leave and Benefits",   "category": "leave_benefits"},
    "3.400": {"title": "Salary and Compensation",                                  "section": "Compensation",         "category": "compensation"},
    "3.500": {"title": "Academic Procedures Overview",                             "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.501": {"title": "Absences",                                                 "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.502": {"title": "Student Records",                                          "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.503": {"title": "Class Hour/Location Changes",                              "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.504": {"title": "Records and Reports",                                      "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.505": {"title": "Field Trips",                                              "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.506": {"title": "Library Services",                                         "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.507": {"title": "Meetings",                                                 "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.508": {"title": "Office Hours",                                             "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.509": {"title": "Problem Resolution",                                       "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.511": {"title": "Student Workers",                                          "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.512": {"title": "Textbook Orders",                                          "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.513": {"title": "Jury Duty",                                                "section": "Academic Procedures",  "category": "academic_procedures"},
    "3.600": {"title": "Resolution of Grievances and Disputes",                    "section": "Grievances",           "category": "grievances"},
    "3.700": {"title": "Faculty Organization and Governance",                      "section": "Governance",           "category": "governance"},
    "3.701": {"title": "Admissions and Re-Entry Committee",                        "section": "Governance",           "category": "governance"},
    "3.702": {"title": "Assessment Committee",                                     "section": "Governance",           "category": "governance"},
    "3.706": {"title": "Faculty Appeals Committee",                                "section": "Governance",           "category": "governance"},
    "3.707": {"title": "Faculty Development Committee",                            "section": "Governance",           "category": "governance"},
    "3.709": {"title": "Graduate Curriculum Committee",                            "section": "Governance",           "category": "governance"},
    "3.710": {"title": "Library Committee",                                        "section": "Governance",           "category": "governance"},
    "3.712": {"title": "Promotion and Tenure Committee",                           "section": "Governance",           "category": "governance"},
    "3.719": {"title": "Undergraduate Curriculum Committee",                       "section": "Governance",           "category": "governance"},
    "4.0":   {"title": "Glossary",                                                 "section": "Appendix",             "category": "appendix"},
}

# ── Regex patterns ────────────────────────────────────────────
_STANDALONE_POLICY_RE = re.compile(r'^((?:3|4)\.\d{1,3})\s*(?:\d+\s*)?$')
EFFECTIVE_DATE_RE     = re.compile(r'(?:Effective Date|Updated)[:\s]+([0-9/\-]+)', re.IGNORECASE)
SECTION_HEADER_RE     = re.compile(
    r'^(?:POLICY|PROCEDURE|PURPOSE|CONDITIONS|DEFINITIONS?|ELIGIBILITY|'
    r'COMPENSATION|BENEFITS?|REQUIREMENTS?|RESPONSIBILITIES|AUTHORITY):\s*$',
    re.IGNORECASE | re.MULTILINE,
)
CROSS_REF_RE    = re.compile(r'\bPolicy\s+(3\.\d+|4\.\d+)', re.IGNORECASE)
NUMBERED_RE     = re.compile(r'^\s*\d+\.\s+\S', re.MULTILINE)
MAX_CHUNK_WORDS = 500
MIN_CHUNK_WORDS = 30


def _detect_policy_num(page_text: str) -> Optional[str]:
    if 'Effective Date:' not in page_text and 'Updated:' not in page_text:
        return None
    for line in page_text.split('\n'):
        m = _STANDALONE_POLICY_RE.match(line.strip())
        if m:
            return m.group(1)
    return None


def _strip_header(page_text: str) -> str:
    idx = page_text.find('Responsible Department:')
    if idx == -1:
        return page_text.strip()
    after = page_text[idx + len('Responsible Department:'):]
    lines, content_lines, skipped = after.split('\n'), [], 0
    for line in lines:
        stripped = line.strip().lower()
        if skipped < 2 and stripped in {'academic affairs','human resources','provost',
                                         'president','student affairs','finance'}:
            skipped += 1; continue
        if skipped < 2 and not stripped:
            skipped += 1; continue
        skipped = 2
        content_lines.append(line)
    return '\n'.join(content_lines).strip()


def _split_sub_chunks(text: str) -> list:
    if len(text.split()) <= MAX_CHUNK_WORDS:
        return [text]
    boundaries = [m.start() for m in SECTION_HEADER_RE.finditer(text)]
    if not boundaries:
        paras = re.split(r'\n{2,}', text)
        chunks, cur = [], []
        for p in paras:
            if len(' '.join(cur).split()) + len(p.split()) > MAX_CHUNK_WORDS and cur:
                chunks.append('\n\n'.join(cur).strip()); cur = [p]
            else:
                cur.append(p)
        if cur: chunks.append('\n\n'.join(cur).strip())
        return [c for c in chunks if c.strip()]
    parts, prev = [], 0
    for b in boundaries:
        if b > prev: parts.append(text[prev:b])
        prev = b
    parts.append(text[prev:])
    result, buf = [], ''
    for p in parts:
        candidate = (buf + '\n\n' + p) if buf else p
        if len(candidate.split()) > MAX_CHUNK_WORDS and buf:
            result.append(buf.strip()); buf = p
        else:
            buf = candidate
    if buf.strip(): result.append(buf.strip())
    return [r for r in result if r.strip()] or [text]


def extract_chunks(pdf_path: str) -> list:
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for p in pdf.pages:
            pages.append(p.extract_text() or '')
    total_words = sum(len(p.split()) for p in pages)
    print(f'Extracted {total_words:,} words across {len(pages)} pages')

    # Pass 1: assign pages to policy numbers
    page_records, cur_policy, cur_date = [], None, None
    for pg_idx, page in enumerate(pages):
        detected = _detect_policy_num(page)
        if detected:
            cur_policy = detected
            d = EFFECTIVE_DATE_RE.search(page)
            cur_date = d.group(1).strip() if d else None
        if cur_policy is None: continue
        content = _strip_header(page)
        if content.strip():
            page_records.append({'policy_num': cur_policy, 'effective_date': cur_date,
                                  'page_num': pg_idx + 1, 'content': content})

    # Pass 2: merge pages per policy
    merged = {}
    for rec in page_records:
        pn = rec['policy_num']
        if pn not in merged:
            merged[pn] = {'policy_num': pn, 'effective_date': rec['effective_date'],
                          'pages': [rec['page_num']], 'content': rec['content']}
        else:
            merged[pn]['pages'].append(rec['page_num'])
            merged[pn]['content'] += '\n\n' + rec['content']

    # Pass 3: split oversized, emit chunks
    chunks, chunk_id = [], 0
    for pn in sorted(merged.keys(), key=lambda x: [int(p) if p.isdigit() else p
                                                    for p in re.split(r'[.]', x)]):
        rec  = merged[pn]
        info = POLICY_MAP.get(pn, {'title': f'Policy {pn}', 'section': 'Unknown', 'category': 'general'})
        for sub_idx, text in enumerate(_split_sub_chunks(rec['content'])):
            wc = len(text.split())
            if wc < MIN_CHUNK_WORDS: continue
            chunks.append({
                'chunk_id': chunk_id, 'policy_num': pn, 'chunk_index': sub_idx,
                'text': text, 'policy_title': info['title'], 'section': info['section'],
                'category': info['category'], 'page_range': rec['pages'],
                'word_count': wc, 'effective_date': rec['effective_date'],
                'has_numbered_list': bool(NUMBERED_RE.search(text)),
                'cross_references': list(set(CROSS_REF_RE.findall(text))),
            })
            chunk_id += 1
    return chunks

print('Chunking functions loaded.')

In [ ]:
if os.path.exists(CHUNKS_PATH):
    chunks = [json.loads(l) for l in open(CHUNKS_PATH)]
    print(f'Loaded {len(chunks)} existing chunks from Drive.')
else:
    chunks = extract_chunks(PDF_PATH)
    with open(CHUNKS_PATH, 'w') as f:
        for c in chunks: f.write(json.dumps(c) + '\n')
    print(f'Saved {len(chunks)} chunks → {CHUNKS_PATH}')

cats = Counter(c['category'] for c in chunks)
print(f'\nTotal chunks : {len(chunks)}')
print(f'Avg words    : {sum(c["word_count"] for c in chunks) // len(chunks)}')
print(f'\n{"Category":<30} {"Chunks":>6}')
print('-' * 38)
for cat, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f'  {cat:<28} {n:>6}')

---
## Step 5 — Generate QA Pairs via Gemma 2

Gemma 2-2B is loaded in 4-bit and used to generate 300 QA pairs from the policy chunks.
Each pair is labeled with one of 8 question types and includes a citation.

Generation runs at **temperature 0.7** for diversity.
The model is unloaded after generation to free VRAM for training.

⚠️ **This step takes ~30–60 minutes on T4.** It is resume-safe — already-generated pairs
are loaded from Drive and new ones are appended.

In [ ]:
from unsloth import FastLanguageModel

gen_model, gen_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(gen_model)
print(f'Model loaded for generation: {MODEL_NAME}')

QUESTION_TYPES = [
    'factual', 'procedural', 'eligibility', 'scenario',
    'rights_responsibilities', 'comparative', 'timeline_deadline', 'committee_role',
]

def generate_qa_pair(chunk: dict, q_type: str) -> Optional[dict]:
    user_content = (
        f"Create one {q_type.replace('_', ' ')} question and answer from this CBU Faculty Manual policy.\n\n"
        f"POLICY: {chunk['policy_num']} — {chunk['policy_title']}\n"
        f"TEXT:\n{chunk['text'][:500]}\n\n"
        f"Rules:\n"
        f"- Question must test specific knowledge from this policy\n"
        f"- Answer must be factual and directly supported by the text\n"
        f"- Answer MUST end with: [Reference: CBU Faculty Manual, Policy {chunk['policy_num']}]\n\n"
        f"Respond in this exact format:\n"
        f"Q: <your question>\n"
        f"A: <your answer ending with the reference>"
    )
    messages = [{"role": "user", "content": user_content}]
    prompt = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(prompt, return_tensors='pt').to('cuda')

    with torch.no_grad():
        out = gen_model.generate(
            **inputs, max_new_tokens=300, temperature=0.7,
            do_sample=True, top_p=0.9,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    text = gen_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    q_m = re.search(r'Q:\s*(.+?)(?=\nA:|$)', text, re.DOTALL)
    a_m = re.search(r'A:\s*(.+)', text, re.DOTALL)
    if not q_m or not a_m:
        return None

    q = q_m.group(1).strip()
    a = a_m.group(1).strip()

    if len(a.split()) < 20:
        return None
    if '[Reference:' not in a:
        a += f'\n\n[Reference: CBU Faculty Manual, Policy {chunk["policy_num"]}]'

    return {
        'question':      q,
        'answer':        a,
        'question_type': q_type,
        'source_policy': chunk['policy_num'],
        'metadata':      {'category': chunk['category'], 'audience': 'all faculty'},
    }

print('Generation function ready.')

In [ ]:
import itertools

TARGET   = 300
existing = [json.loads(l) for l in open(SEEDS_PATH)] if os.path.exists(SEEDS_PATH) else []
print(f'Existing QA pairs: {len(existing)}')

seeds_file = open(SEEDS_PATH, 'a', encoding='utf-8')
generated  = len(existing)
q_type_cycle = itertools.cycle(QUESTION_TYPES)
chunk_cycle  = itertools.cycle(chunks)
filtered     = 0

while generated < TARGET:
    chunk  = next(chunk_cycle)
    q_type = next(q_type_cycle)
    pair   = generate_qa_pair(chunk, q_type)

    if pair is None:
        filtered += 1
        continue

    seeds_file.write(json.dumps(pair) + '\n')
    seeds_file.flush()
    generated += 1

    if generated % 10 == 0 or generated == TARGET:
        print(f'  [{generated}/{TARGET}] filtered so far: {filtered}')

seeds_file.close()
print(f'\nDone. {generated} QA pairs saved → {SEEDS_PATH}')

In [ ]:
# Unload generation model before training
import gc
del gen_model, gen_tokenizer
gc.collect()
torch.cuda.empty_cache()
print('Generation model unloaded. VRAM freed.')

seeds = [json.loads(l) for l in open(SEEDS_PATH)]
type_counts = Counter(s['question_type'] for s in seeds)
print(f'\nTotal QA pairs : {len(seeds)}')
print(f'\n{"Question Type":<28} {"Count":>5}')
print('-' * 35)
for qt, n in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {qt:<26} {n:>5}')

print('\n── Sample QA pair ──')
s = seeds[0]
print(f'Type     : {s["question_type"]}')
print(f'Question : {s["question"]}')
print(f'Answer   : {s["answer"][:200]}...')

---
## Step 6 — Format Dataset

QA pairs are wrapped in Gemma 2's chat template and split 90/10 into train and validation sets.

Gemma 2 training format:
```
<bos><start_of_turn>user
[system prompt]\n\n[question]<end_of_turn>
<start_of_turn>model
[answer]<end_of_turn>
```

In [ ]:
import random
from transformers import AutoTokenizer

fmt_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

SYSTEM_PROMPT = (
    "You are a knowledgeable CBU Faculty Affairs assistant with expertise in the "
    "California Baptist University Employee Manual — Faculty Section (Updated 01/2026). "
    "You help faculty understand university policies, procedures, rights, and responsibilities. "
    "Place all policy citations at the END of your answer in "
    "[Reference: CBU Faculty Manual, Policy X.XXX] format — NEVER at the beginning. "
    "If a question cannot be answered from the CBU Faculty Manual, say so clearly."
)

CATEGORY_LABELS = {
    'foundation': 'CBU Mission and Values', 'academic_freedom': 'Academic Freedom',
    'employment': 'Faculty Employment', 'conduct': 'Faculty Conduct',
    'tenure_promotion': 'Tenure and Promotion', 'leave_benefits': 'Leave and Benefits',
    'compensation': 'Compensation', 'academic_procedures': 'Academic Procedures',
    'grievances': 'Grievances', 'governance': 'Governance', 'general': 'Faculty Policy',
}

def format_example(seed: dict):
    q    = seed.get('question', '').strip()
    a    = seed.get('answer',   '').strip()
    meta = seed.get('metadata', {})
    if not q or not a:
        return None

    cat    = meta.get('category', 'general')
    label  = CATEGORY_LABELS.get(cat, 'Faculty Policy')
    aud    = meta.get('audience', 'all faculty').replace('_', ' ')
    system = f"{SYSTEM_PROMPT} Topic area: {label}. Audience: {aud}."

    messages = [
        {"role": "user",      "content": f"{system}\n\n{q}"},
        {"role": "assistant", "content": a},
    ]
    text = fmt_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {'text': text, 'question_type': seed.get('question_type')}

examples = [e for s in seeds if (e := format_example(s)) is not None]
random.seed(42)
random.shuffle(examples)

n_val   = max(1, int(len(examples) * 0.10))
n_train = len(examples) - n_val
train_examples = examples[:n_train]
valid_examples = examples[n_train:]

with open(TRAIN_PATH, 'w') as f:
    for ex in train_examples: f.write(json.dumps(ex) + '\n')
with open(VALID_PATH, 'w') as f:
    for ex in valid_examples: f.write(json.dumps(ex) + '\n')

avg_tokens = sum(len(e['text'].split()) for e in train_examples) // len(train_examples)
print(f'Train : {n_train} examples')
print(f'Valid : {n_val} examples')
print(f'Avg   : ~{avg_tokens} words per example')
print(f'Saved → {TRAIN_PATH}')

del fmt_tokenizer

---
## Step 7 — QLoRA Fine-Tuning

Gemma 2-2B is fine-tuned using QLoRA — the base weights stay frozen in 4-bit quantization
and only the low-rank adapter matrices (r=16) are trained.

| Parameter | Value |
|-----------|-------|
| Base model | `unsloth/gemma-2-2b-it` |
| Method | QLoRA (rank 16, α 16) |
| Epochs | 3 |
| Learning rate | 2e-4 |
| Batch size | 2 + gradient accumulation 4 |
| Optimizer | AdamW 8-bit |

⚠️ **This step takes ~30–45 minutes on T4.** Checkpoints are saved every 100 steps to Drive.
If your session disconnects, re-run this cell — training resumes from the latest checkpoint.

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

train_rows = [json.loads(l) for l in open(TRAIN_PATH)]
train_dataset = Dataset.from_list([{'text': r['text']} for r in train_rows])
print(f'Training on {len(train_dataset)} examples')

CKPT_DIR = f'{DRIVE_BASE}/outputs/checkpoints'

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        output_dir=CKPT_DIR,   # saved to Drive — survives session disconnect
        save_steps=100,
        save_total_limit=3,
    ),
)

trainer_stats = trainer.train()
print(f'\nTraining complete. Runtime: {trainer_stats.metrics["train_runtime"]/60:.1f} min')

---
## Step 8 — Save Adapter & Export GGUF to Drive

In [ ]:
# Save LoRA adapter to Drive
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'Adapter saved → {ADAPTER_PATH}')

# Export Q4_K_M GGUF directly from training session
GGUF_LOCAL = '/content/gguf_model'
model.save_pretrained_gguf(GGUF_LOCAL, tokenizer, quantization_method='q4_k_m')
print(f'GGUF export complete')

# Copy GGUF to Drive
import glob, shutil
gguf_files = glob.glob(f'{GGUF_LOCAL}/*.gguf')
for f in gguf_files:
    dst = f'{GGUF_DIR}/{os.path.basename(f)}'
    shutil.copy(f, dst)
    size_gb = os.path.getsize(dst) / 1e9
    print(f'GGUF saved → {dst}  ({size_gb:.2f} GB)')

# Write Modelfile for Ollama
gguf_name = os.path.basename(gguf_files[0]) if gguf_files else 'model.Q4_K_M.gguf'
modelfile_content = (
    f'FROM ./{gguf_name}\n\n'
    f'SYSTEM """You are a knowledgeable CBU Faculty Affairs assistant with expertise in the '
    f'California Baptist University Employee Manual — Faculty Section (Updated 01/2026). '
    f'You help faculty understand university policies, procedures, rights, and responsibilities. '
    f'Place all policy citations at the END of your answer in '
    f'[Reference: CBU Faculty Manual, Policy X.XXX] format — NEVER at the beginning. '
    f'If a question cannot be answered from the CBU Faculty Manual, say so clearly."""\n\n'
    f'PARAMETER temperature 0.1\n'
    f'PARAMETER top_p 0.9\n'
    f'PARAMETER num_ctx 2048\n'
    f'PARAMETER num_predict 512\n'
)
with open(f'{GGUF_DIR}/Modelfile', 'w') as f:
    f.write(modelfile_content)
print(f'Modelfile written → {GGUF_DIR}/Modelfile')
print('\nDeploy with Ollama:')
print(f'  ollama create cbu-faculty-gemma2 -f {GGUF_DIR}/Modelfile')
print('  ollama run cbu-faculty-gemma2')

---
## Step 9 — Evaluate ROUGE-L

ROUGE-L measures word-sequence overlap between model-generated answers and reference answers.
Both base model and fine-tuned model are evaluated on the same 30 validation questions.

⚠️ **If the runtime was restarted**, re-run Steps 0–2 first (install, GPU check, drive mount),
then continue here — the adapter is loaded directly from Drive.

In [ ]:
import gc
import re
import torch
import json
import os
import numpy as np
from unsloth import FastLanguageModel
from rouge_score import rouge_scorer as rs_module

# Re-establish paths if runtime was restarted after Step 8
if 'DRIVE_BASE' not in dir():
    DRIVE_BASE   = '/content/drive/MyDrive/cbu-faculty-llm'
    VALID_PATH   = f'{DRIVE_BASE}/data/valid.jsonl'
    ADAPTER_PATH = f'{DRIVE_BASE}/outputs/cbu_gemma2_adapter'
    EVAL_PATH    = f'{DRIVE_BASE}/eval/results.jsonl'
    MODEL_NAME   = 'unsloth/gemma-2-2b-it'

valid_rows = [json.loads(l) for l in open(VALID_PATH)]
scorer     = rs_module.RougeScorer(['rougeL'], use_stemmer=True)


def extract_qa(row: dict):
    text = row['text']
    q_m = re.search(r'<start_of_turn>user\n(.+?)<end_of_turn>', text, re.DOTALL)
    a_m = re.search(r'<start_of_turn>model\n(.+?)<end_of_turn>', text, re.DOTALL)
    if not q_m or not a_m:
        return None, None
    return q_m.group(1).strip(), a_m.group(1).strip()


def run_inference(model, tokenizer, question: str, max_new_tokens=300) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.1, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


questions, references, qtypes = [], [], []
for row in valid_rows:
    q, ref = extract_qa(row)
    if q and ref:
        questions.append(q)
        references.append(ref)
        qtypes.append(row.get('question_type', 'unknown'))
print(f'Parsed {len(questions)} validation examples')

# ── Base model ────────────────────────────────────────────────
print('\nLoading base model...')
base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=2048, dtype=None, load_in_4bit=True
)
FastLanguageModel.for_inference(base_model)

base_preds = []
for i, q in enumerate(questions, 1):
    base_preds.append(run_inference(base_model, base_tok, q))
    if i % 5 == 0: print(f'  [base] {i}/{len(questions)}')

del base_model, base_tok
gc.collect(); torch.cuda.empty_cache()
print('Base model unloaded.')

# ── Fine-tuned model (load adapter directly via Unsloth) ──────
print('\nLoading fine-tuned model...')
ft_model, ft_tok = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,   # Unsloth reads adapter_config.json and loads base + adapter
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)

ft_preds = []
for i, q in enumerate(questions, 1):
    ft_preds.append(run_inference(ft_model, ft_tok, q))
    if i % 5 == 0: print(f'  [ft]   {i}/{len(questions)}')

del ft_model, ft_tok
gc.collect(); torch.cuda.empty_cache()
print('Fine-tuned model unloaded.')

In [ ]:
import numpy as np

results = []
for q, ref, pred_b, pred_ft, qt in zip(questions, references, base_preds, ft_preds, qtypes):
    b_score  = scorer.score(ref, pred_b)['rougeL'].fmeasure
    ft_score = scorer.score(ref, pred_ft)['rougeL'].fmeasure
    results.append({
        'question': q[:80], 'question_type': qt,
        'base_rougeL': round(b_score, 4),
        'ft_rougeL':   round(ft_score, 4),
        'delta':       round(ft_score - b_score, 4),
    })

os.makedirs(os.path.dirname(EVAL_PATH), exist_ok=True)
with open(EVAL_PATH, 'w') as f:
    for r in results: f.write(json.dumps(r) + '\n')

base_scores = [r['base_rougeL'] for r in results]
ft_scores   = [r['ft_rougeL']   for r in results]
improved    = sum(1 for b, f in zip(base_scores, ft_scores) if f > b)
pct         = (np.mean(ft_scores) - np.mean(base_scores)) / np.mean(base_scores) * 100

print('=' * 50)
print('  ROUGE-L Evaluation Results')
print('=' * 50)
print(f'  Examples evaluated : {len(results)}')
print(f'  Base model avg     : {np.mean(base_scores):.4f}')
print(f'  Fine-tuned avg     : {np.mean(ft_scores):.4f}')
print(f'  Improvement        : {pct:+.1f}%')
print(f'  Examples improved  : {improved}/{len(results)}')
print('=' * 50)

---
## Step 10 — Plot Results by Question Type

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from IPython.display import Image, display

TYPE_LABELS = {
    'factual': 'Factual', 'procedural': 'Procedural',
    'eligibility': 'Eligibility', 'scenario': 'Scenario',
    'rights_responsibilities': 'Rights &\nResponsibilities',
    'comparative': 'Comparative', 'timeline_deadline': 'Timeline /\nDeadline',
    'committee_role': 'Committee\nRole',
}
TYPE_ORDER = list(TYPE_LABELS.keys())

buckets = defaultdict(lambda: {'base': [], 'ft': []})
for r in results:
    buckets[r['question_type']]['base'].append(r['base_rougeL'])
    buckets[r['question_type']]['ft'].append(r['ft_rougeL'])

ordered = [qt for qt in TYPE_ORDER if qt in buckets]
ordered += [qt for qt in buckets if qt not in ordered]

labels    = [TYPE_LABELS.get(qt, qt) for qt in ordered]
base_vals = [np.mean(buckets[qt]['base']) for qt in ordered]
ft_vals   = [np.mean(buckets[qt]['ft'])   for qt in ordered]
n_vals    = [len(buckets[qt]['base'])      for qt in ordered]

x, width = np.arange(len(ordered)), 0.35
fig, ax  = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

bars_b  = ax.bar(x - width/2, base_vals, width, label='Base Model',      color='#4a90d9', alpha=0.85, zorder=3)
bars_ft = ax.bar(x + width/2, ft_vals,   width, label='Fine-tuned (CBU)', color='#e05c5c', alpha=0.85, zorder=3)

for bar in bars_b:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
            ha='center', va='bottom', fontsize=8, color='#aaaaaa')
for bar in bars_ft:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
            ha='center', va='bottom', fontsize=8, color='white', fontweight='bold')
for i, n in enumerate(n_vals):
    ax.text(x[i], -0.025, f'n={n}', ha='center', va='top', fontsize=7.5, color='#888888')

ax.set_xlabel('Question Type', color='#cccccc', fontsize=11, labelpad=24)
ax.set_ylabel('ROUGE-L Score', color='#cccccc', fontsize=11)
ax.set_title(
    'ROUGE-L: Base Model vs Fine-tuned (CBU Faculty Manual)\n'
    f'{len(results)} validation examples  ·  Gemma 2-2B  ·  300 QA pairs  ·  3 epochs',
    color='white', fontsize=12, fontweight='bold', pad=14,
)
ax.set_xticks(x)
ax.set_xticklabels(labels, color='#cccccc', fontsize=9)
ax.set_ylim(0, max(ft_vals + base_vals) * 1.25)
ax.tick_params(colors='#888888', which='both')
for spine in ax.spines.values(): spine.set_edgecolor('#333333')
ax.grid(axis='y', color='#333333', linestyle='--', alpha=0.6, zorder=0)

overall_base = np.mean(base_vals)
overall_ft   = np.mean(ft_vals)
pct_leg      = (overall_ft - overall_base) / overall_base * 100
patch_b  = mpatches.Patch(color='#4a90d9', alpha=0.85, label=f'Base Model  (avg {overall_base:.4f})')
patch_ft = mpatches.Patch(color='#e05c5c', alpha=0.85, label=f'Fine-tuned  (avg {overall_ft:.4f}  ·  {pct_leg:+.1f}%)')
ax.legend(handles=[patch_b, patch_ft], facecolor='#1a1d27', edgecolor='#444444',
          labelcolor='#dddddd', fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig(CHART_PATH, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
print(f'Chart saved → {CHART_PATH}')
plt.close()

display(Image(CHART_PATH))

---
## Summary

| Stage | Output | Notes |
|-------|--------|-------|
| Chunk | ~90 policy sections | 11 categories, avg ~290 words |
| Generate | 300 QA pairs | 8 question types, LLM-generated |
| Format | 270 train / 30 valid | Gemma 2 chat template |
| Train | QLoRA adapter | 3 epochs, r=16, lr=2e-4 |
| Evaluate | ROUGE-L | See chart above |
| Export | Q4_K_M GGUF | Ready for Ollama |

**Differences from the MLX (Apple Silicon) version:**

| | This notebook (Colab) | MLX notebook (Mac) |
|---|---|---|
| Model | Gemma 2-2B | Gemma 4 E4B |
| Method | QLoRA (quantized base) | LoRA (pre-quantized base) |
| QA generation | Model self-inference | Ollama (separate process) |
| Reasoning traces | No (Gemma 2 format) | Yes (`<\|channel>thought`) |
| Hardware | T4 GPU (16 GB VRAM) | Apple Silicon (unified memory) |
| Portability | Any browser + Google account | Mac M-series only |

**Recommendations for improvement:**
- Monitor val loss across epochs — use the epoch with lowest val loss, not necessarily the final one
- Human evaluation on top of ROUGE-L — it does not catch hallucinated policy numbers
- Expand QA pairs targeting under-represented types (Comparative, Committee Role)